In [1]:
import pandas as pd
import os
from tqdm import tqdm
import json
import os
import subprocess
from itertools import product

DO_ALL_MEETINGS = True
# AMI_PATH = '/home/sgoyal/Projects/llm_asr_clarification/shared/datasets/amicorpus/train'
AMI_PATH = '/home/pkongsomjit/Projects/llm_asr_clarification/shared/datasets/amicorpus/validation'
# AMI_PATH = '/group/jrwhitehill/amicorpus'
# AMI_PATH = '/home/surigo/Projects/WPI-Research-Projects/Automatic-Speech-Recognition/Code/llm_asr_clarification/datasets/amicorpus'
# AMI_PATH = '/home/pkongsomjit/Projects/llm_asr_clarification/datasets/amicorpus/xinlu_data'
MEETING_TO_DO= '/home/pkongsomjit/Projects/llm_asr_clarification/datasets/amicorpus/ES2005d'
# QUESTION_FILE = 'parsed_gt'
QUESTION_FILE = 'parsed_diarized_gt'

ARGUMENTS = {
    # 'transcript_files' : [
    #     # 'whisper_tiny_diarized_transcript',
    #     # 'whisper_tiny_diarized_transcript_random_clarify',
    #     # 'whisper_tiny_diarized_transcript_llm-orig-ctx_clarify',
    #     # 'whisper_tiny_diarized_transcript_llm-gt-ctx_clarify',
    #     # 'whisper_tiny_diarized_transcript_llm-orig-ctx_clarify_sample2',
    #     # 'whisper_tiny_diarized_transcript_llm-gt-ctx_clarify_sample2',
    #     # 'qwen_transcript',
    #     # 'tiny_transcript',
    #     # 'large_transcript',
    #     # 'whisper-large-v3_transcript',
    #     # 'whisper-tiny_transcript',
    #     # 'parsed_diarized_gt',
    #     'custom_transcript_gt_segments',
    #     # 'custom_transcript_gt_segments_gt_clarify',
    #     # 'custom_transcript_gt_segments_random_clarify',
    #     # 'custom_transcript_gt_segments_rf_clarify',
    #     # 'custom_transcript_gt_segments_clarify_only_importance',
    #     # 'custom_transcript_gt_segments_gt_clarify2',
    #     # 'custom_transcript_gt_segments_noise',
    #     # 'custom_transcript_gt_segments_all_gt_clarify3',
    #     # 'custom_transcript_gt_segments_all_lstm_clarify3',
    #     # 'custom_transcript_gt_segments_gt_lstm_clarify3',
    # ],
    'splits' : ['validation'],
    'clarification_num_lines': ["10", "20", "30", "40", "50"],
    'importance_detectors' : ["LSTM", "GT"],
    'mistranscript_detectors' : ["RF", "GT", "ALL"]
}

MODEL_TO_USE = 'gpt-4o-mini' #'gpt-5.4-mini'

keys = ARGUMENTS.keys()
values = ARGUMENTS.values()

# 2. Compute the Cartesian product and rebuild dictionaries
combinations = [dict(zip(keys, v)) for v in product(*values)]

TRANSCRIPT_FILES = ['parsed_diarized_gt', 'custom_transcript_gt_segments']
for arg in combinations:

    split = arg['splits']
    clarification_num_line = arg['clarification_num_lines']
    importance_detector = arg['importance_detectors']
    mistranscript_detector = arg['mistranscript_detectors']

    TRANSCRIPT_FILES.append(f"custom_transcript_gt_segments_{mistranscript_detector.lower()}_{importance_detector.lower()}_{clarification_num_line}_clarify3")

TRANSCRIPT_FILES = [f'score_using_{t}' for t in TRANSCRIPT_FILES]

# directories of meetings
if DO_ALL_MEETINGS:
    meeting_paths = [entry.path for entry in os.scandir(AMI_PATH)]

list_of_quiz_dicts = []
for meeting_path in tqdm(meeting_paths):
    question_path = os.path.join(meeting_path, "quiz", f"quiz_from_{QUESTION_FILE}.json")
    # question_path = os.path.join(meeting_path, "quiz", "old_quiz.json")
        
    # chatgpt = OpenAIWrapper()
    
    # Read question
    try:
        with open(question_path, "r", encoding="utf-8") as f:
            quiz = f.read()

        quiz = json.loads(quiz)
        meeting_name = meeting_path.split("/")[-1]
        for q in quiz:
            q['meeting_name'] = meeting_name
        list_of_quiz_dicts += quiz
    except Exception as err:
        print(f"couldnt open file {question_path}")

df = pd.DataFrame(list_of_quiz_dicts)

100%|██████████| 18/18 [00:00<00:00, 3269.56it/s]


In [2]:
df.head()

,question,correct_answer,answer_using_qwen_transcript,score_using_qwen_transcript,answer_using_whisper_tiny_diarized_transcript,score_using_whisper_tiny_diarized_transcript,answer_using_parsed_diarized_gt,score_using_parsed_diarized_gt,answer_using_custom_transcript_gt_segments_gt_lstm_clarify3,answer_using_custom_transcript_gt_segments_all_gt_clarify3,...,score_using_custom_transcript_gt_segments_gt_lstm_40_clarify3,score_using_custom_transcript_gt_segments_all_lstm_40_clarify3,score_using_custom_transcript_gt_segments_rf_lstm_20_clarify3,score_using_custom_transcript_gt_segments_all_lstm_50_clarify3,score_using_custom_transcript_gt_segments_rf_lstm_40_clarify3,score_using_custom_transcript_gt_segments_rf_lstm_50_clarify3,score_using_custom_transcript_gt_segments_gt_lstm_30_clarify3,score_using_custom_transcript_gt_segments_rf_lstm_30_clarify3,score_using_custom_transcript_gt_segments_gt_lstm_50_clarify3,meeting_name
0,What was the main reason the team decided to c...,Because the fruit-shaped versions were awkward...,The team decided to change the remote's shape ...,1.0,To ensure it was hand-holdable and comfortable...,n/a,The team decided to change the remote's shape ...,1,The team decided to change the remote's shape ...,The team decided to change the remote's shape ...,...,1,1,1,1,1,1,1,1,1,ES2011d
1,What did the team conclude about the relations...,A round or bulky fruit shape made the remote l...,The team concluded that the shape of the remot...,1.0,The shape of the remote significantly affects ...,1,The team concluded that the shape of the remot...,1,The team concluded that the shape of the remot...,The team concluded that the shape of the remot...,...,1,1,1,1,1,1,1,1,1,ES2011d
2,What was the key issue with the evaluation sca...,They had reversed the scale and initially thou...,The key issue with the evaluation scale was th...,1.0,There was confusion about what the scale repre...,1,The key issue with the evaluation scale was th...,1,The key issue with the evaluation scale was th...,The key issue with the evaluation scale was co...,...,1,1,1,1,1,1,1,1,1,ES2011d
3,According to the corrected evaluation criteria...,1 meant true and 7 meant false.,"In the corrected evaluation criteria, the numb...",1.0,"The number 1 represented 'true' or 'fancy', wh...",0,According to the corrected evaluation criteria...,1,According to the corrected evaluation criteria...,According to the corrected evaluation criteria...,...,0,0,0,0,0,0,0,0,0,ES2011d
4,Which evaluation criterion was identified as t...,"Material being technologically innovative, spe...",The weakest evaluation criterion identified wa...,1.0,The weakest evaluation criterion identified wa...,1,The weakest evaluation criterion identified wa...,1,The weakest evaluation criterion identified wa...,The weakest evaluation criterion identified wa...,...,0,0,0,0,0,0,0,0,0,ES2011d


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 77 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   question                                                         180 non-null    str    
 1   correct_answer                                                   180 non-null    str    
 2   answer_using_qwen_transcript                                     140 non-null    str    
 3   score_using_qwen_transcript                                      50 non-null     float64
 4   answer_using_whisper_tiny_diarized_transcript                    180 non-null    str    
 5   score_using_whisper_tiny_diarized_transcript                     180 non-null    object 
 6   answer_using_parsed_diarized_gt                                  180 non-null    str    
 7   score_using_parsed_diarized_gt                         

In [4]:
for transcript_file in TRANSCRIPT_FILES:
    df[transcript_file] = pd.to_numeric(df[transcript_file], errors="coerce")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 77 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   question                                                         180 non-null    str    
 1   correct_answer                                                   180 non-null    str    
 2   answer_using_qwen_transcript                                     140 non-null    str    
 3   score_using_qwen_transcript                                      50 non-null     float64
 4   answer_using_whisper_tiny_diarized_transcript                    180 non-null    str    
 5   score_using_whisper_tiny_diarized_transcript                     180 non-null    object 
 6   answer_using_parsed_diarized_gt                                  180 non-null    str    
 7   score_using_parsed_diarized_gt                         

In [5]:
for transcript_file in TRANSCRIPT_FILES:
    print(transcript_file)
    print(df[transcript_file].mean())

score_using_parsed_diarized_gt
0.9209039548022598
score_using_custom_transcript_gt_segments
0.7888888888888889
score_using_custom_transcript_gt_segments_rf_lstm_10_clarify3
0.8055555555555556
score_using_custom_transcript_gt_segments_gt_lstm_10_clarify3
0.8111111111111111
score_using_custom_transcript_gt_segments_all_lstm_10_clarify3
0.7666666666666667
score_using_custom_transcript_gt_segments_rf_gt_10_clarify3
0.8166666666666667
score_using_custom_transcript_gt_segments_gt_gt_10_clarify3
0.8333333333333334
score_using_custom_transcript_gt_segments_all_gt_10_clarify3
0.8111111111111111
score_using_custom_transcript_gt_segments_rf_lstm_20_clarify3
0.8333333333333334
score_using_custom_transcript_gt_segments_gt_lstm_20_clarify3
0.8333333333333334
score_using_custom_transcript_gt_segments_all_lstm_20_clarify3
0.8277777777777777
score_using_custom_transcript_gt_segments_rf_gt_20_clarify3
0.8166666666666667
score_using_custom_transcript_gt_segments_gt_gt_20_clarify3
0.8333333333333334
score

In [6]:
import re
import pandas as pd

rows = []

for col in TRANSCRIPT_FILES:
    score = df[col].mean()

    # Baselines
    if col == "score_using_parsed_diarized_gt":
        rows.append(["Baseline parsed diarized GT", None, None, None, None, score])

    elif col == "score_using_custom_transcript_gt_segments":
        rows.append(["Baseline custom GT segments", None, None, None, None, score])

    # Experimental conditions
    else:
        match = re.search(r"_(rf|gt|all)_(lstm|gt)_(10|20|30|40|50)_clarify3$", col)

        if match:
            model, target, n = match.groups()
            rows.append([f"{model.upper()} + {target.upper()}", int(n), score])

# Easier approach: build directly into a dictionary
results = {}

for col in TRANSCRIPT_FILES:
    score = df[col].mean()

    if col == "score_using_parsed_diarized_gt":
        # results["Baseline parsed diarized GT"] = {"10": score}
        pass
    elif col == "score_using_custom_transcript_gt_segments":
        # results["Baseline custom GT segments"] = {"10": score}
        pass
    else:
        match = re.search(
            r"_(rf|gt|all)_(lstm|gt)_(10|20|30|40|50)_clarify3$",
            col
        )

        if match:
            model, target, n = match.groups()
            key = f"{model.upper()} + {target.upper()}"
            results.setdefault(key, {})[n] = score

table = pd.DataFrame(results).T
table = table[["10", "20", "30", "40", "50"]]

# Pretty print
print(table.round(3).to_string())

               10     20     30     40     50
RF + LSTM   0.806  0.833  0.839  0.817  0.794
GT + LSTM   0.811  0.833  0.794  0.856  0.811
ALL + LSTM  0.767  0.828  0.811  0.806  0.839
RF + GT     0.817  0.817  0.828  0.844  0.844
GT + GT     0.833  0.833  0.828  0.850  0.833
ALL + GT    0.811  0.861  0.856  0.867  0.878


In [6]:
from scipy.stats import f_oneway, ttest_rel

stat, pvalue = f_oneway(
    *[df[t] for t in TRANSCRIPT_FILES]
)

print(f"Stat: {stat}")
print(f"P Value: {pvalue}")

Stat: nan
P Value: nan


In [13]:
import numpy as np

stat, pvalue = ttest_rel(
    df["score_using_custom_transcript_gt_segments"],
    df["score_using_custom_transcript_gt_segments_gt_lstm_clarify3"],
    alternative="less"
)

print(f"Stat: {stat}")
print(f"P Value: {pvalue}")

Stat: -1.6914646409484733
P Value: 0.045488682031720996


In [ ]:
for transcript_file in TRANSCRIPT_FILES:
    print(transcript_file)
    print(df[transcript_file].value_counts())

In [ ]:
from scipy.stats import paired_ttest
p, v = paired_ttest(df['

In [ ]:
stats_dict = {}
for transcript_file in TRANSCRIPT_FILES:
    print(f"Grouped stats for {transcript_file}")
    stats = df.groupby("meeting_name")[transcript_file].describe()
    print(stats)
    stats_dict[transcript_file] = stats

In [ ]:
for transcript_file in TRANSCRIPT_FILES:
    print(f"Global stats by group for {transcript_file}")
    print(f"Mean across groups: {stats_dict[transcript_file]['mean'].mean()}")
    print(f"STD across groups: {stats_dict[transcript_file]['mean'].std()}")
    print(f"Min across groups: {stats_dict[transcript_file]['mean'].min()}")
    print(f"Max across groups: {stats_dict[transcript_file]['mean'].max()}")
    print()